# VCBHN — Sign-in baseline notebook (Sentinel data lake)

Runs in VS Code with the **Microsoft Sentinel** extension on a Sentinel Spark pool.

**Goal:** build a 30-day per-user baseline of sign-in countries and flag users who signed in from a country never seen before.

> Check the provider class name and methods against the current Microsoft Learn page *Sentinel data lake notebooks* before running; the API is new and may change.

In [ ]:
from sentinel_lake.providers import MicrosoftSentinelProvider
from pyspark.sql import functions as F

provider = MicrosoftSentinelProvider(spark)
WORKSPACE = "law-vcbhn-sentinel"  # your workspace name
signins = provider.read_table("SigninLogs", WORKSPACE)

In [ ]:
recent = signins.filter(F.col("TimeGenerated") >= F.date_sub(F.current_date(), 30))
with_country = recent.withColumn("Country", F.col("LocationDetails.countryOrRegion"))

baseline = (with_country
    .filter(F.col("TimeGenerated") < F.date_sub(F.current_date(), 1))
    .groupBy("UserPrincipalName")
    .agg(F.collect_set("Country").alias("KnownCountries")))

today = with_country.filter(F.col("TimeGenerated") >= F.date_sub(F.current_date(), 1))

In [ ]:
new_country = (today.join(baseline, "UserPrincipalName", "left")
    .filter(~F.array_contains(F.coalesce(F.col("KnownCountries"), F.array()), F.col("Country")))
    .select("TimeGenerated", "UserPrincipalName", "IPAddress", "Country", "AppDisplayName"))

new_country.show(50, truncate=False)

## Findings

Record what you found here (sanitized), and whether it should become a scheduled notebook job or a KQL detection.